# ChEMBL
- 화합물 데이터베이스 중 하나입니다. 특정 화합물이나 증상같은걸로 검색하면 관련된 화합물을 찾아주는 서비스입니다.
- 검색 결과를 csv 파일로 저장할 수 있습니다.
- 뭐 가져오는 라이브러리도 있는 것 같은데 오늘은 예전에 받아뒀던거 걍 쓸겁니다. 

## 프로젝트 정보
- 인원: 1인
- 파이썬 버전: 3.10
- 데이터 리소스: ChEMBL
- 필요한 모듈: Numpy, Pandas, Matplotlib, Seaborn

In [ ]:
# 모! 듈! 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from chembl_webresource_client.new_client import new_client # 켐블털이 
from scipy import stats # 통계분석용

# 그래프 기본 테마 설정
sns.set_theme(palette="viridis", style="whitegrid", font_scale=1) # 블루톤

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Galmuri14' # 제가... 픽셀체 이런거 좋아해서... 
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['axes.titlesize'] = 16 # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14 # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
data_df = pd.read_csv("data/Ibuprofen.csv", sep=";") # 구분자가 ;
data_df

# 데이터 정보 확인

## df.shape

In [ ]:
data_df.shape

## df.info()

In [ ]:
data_df.info() # 얼마 되지도 않는거에 널까지 껴있네 

## df.describe()

In [ ]:
data_df.describe()
# 여기 나왔어도 범주형인것도 있습니다. 

In [ ]:
data_df.describe(include="O")

## df.isna().sum()

In [ ]:
data_df.isna().sum() 
# 아 널이예요... 
# 근데 이 널은 우리가 몰라서 못 채워요... 

## df.head()

In [ ]:
data_df.head()

## df.columns

In [ ]:
data_df.columns

# 전처리

## 분자량 범주화
- 500보다 큰가, 작은가로 나눌거다. 
- 왜냐고요? RO5(리핀스키의 Rule of 5)에 따라 500보다 큰 분자는 거의 약이 아니기 때문임. 

In [ ]:
# 분자량이 500보다 크면 헤비 아니면 라이트
data_df['Moecular_classification'] = data_df['Molecular Weight'].apply(lambda x: 'Heavy' if x > 500 else 'Light') 

## Max Phase 범주화
- 4 아니면 다 임상중인겁니다. -1은... 어... 힘내라... 

In [ ]:
# 4: Approved/3~0: Clinical/-1: Failed
data_df['Status'] = data_df['Max Phase'].map({4: 'Approved', 3: 'Clinical', 2: 'Clinical', 1: 'Clinical', 0: 'Clinical', -1: 'Failed'}) 
# 아니 맵합수가 왜 여기서

In [ ]:
data_df['Status'].value_counts()

# 본게임은 지금부터다

## Max phase별로 보기

### Max phase에 따른 상태별 분자량 평균

In [ ]:
data_df.groupby('Status')['Molecular Weight'].mean() # 평균

In [ ]:
# boxplot
sns.boxplot(data_df, x = 'Status', y = 'Molecular Weight', hue = 'Status')
sns.swarmplot(data=data_df, x='Status', y='Molecular Weight', color='red', alpha=0.5)
plt.title('Molecular Weight by Status')
plt.show()

- Approved에 저 이상치 뭐냐? 500을 넘겼네? 
- Clinical은 아직 임상 진행중인 약물들이라 배리에이션이 다양하고, Failed는 어... 모종의 이유로 엎은거다. 분자량 때문은 아닌 듯 하지만. 

### 이상치가 있네?

In [ ]:
data_df.query('`Molecular Weight` > 500').index # 40번이 나왔다
data_df[['Name','Molecular Weight','Molecular Formula', 'Smiles']].loc[40] # FENOPROFEN CALCIUM (분자량 558.64)

- FENOPROFEN CALCIUM에 대해 찾아보니 쟤가 구조상 카복실산이 달려있음. 
- 칼슘이 옆구리에 쟤를 하나씩 끼고 있어서 분자량이 커진거지, 개별 분자량은 작은 편. 

In [ ]:
# clinical에도 이상치가 있더라? 
idx_list = data_df.query('`Molecular Weight` > 400 and Status == "Clinical"').index # 2, 13, 14, 34
for idx in idx_list:
    display(data_df[['Name','Molecular Weight','Molecular Formula', 'Smiles','Max Phase']].loc[idx])

- 임상 단계에 있는 약물중에는 뭐 붙어있는 애들이 좀 많다. 라이신 뭐 이런거. 

In [ ]:
# 실패한 칭구칭긔들
idx_list = data_df.query('Status == "Failed"').index # 23, 27
for idx in idx_list:
    display(data_df[['Name','Molecular Weight','Molecular Formula', 'Smiles','Max Phase']].loc[idx])

## Bioactivity

In [ ]:
data_df.groupby('Status')['Bioactivities'].mean() # 평균

In [ ]:
# boxplot
sns.boxplot(data_df, x = 'Status', y = 'Bioactivities', hue = 'Status')
sns.swarmplot(data=data_df, x='Status', y='Bioactivities', color='red', alpha=0.5)
plt.title('Bioactivities by Status')
plt.show()

- 저 4000 저거 누구냐. 

### 이상치 범인 찾기

In [ ]:
data_df.query('Bioactivities > 4000').index # 50
data_df[['Name','Molecular Weight','Molecular Formula', 'Bioactivities', 'Smiles']].loc[50] # IBUPROFEN

- 아... 원조가 제일 쎄네... 

### 맨 휘트니 분석
- 저기 Clinical하고 Failed간에 차이가 있는지를 한번 검정해보자. 

In [ ]:
# 맨 휫흐니 검정을 할거예요 
# 얘네는 수가 적어서 t-test 못해요 
clinical_vals = data_df[data_df['Status'] == 'Clinical']['Bioactivities'].dropna() # 클리니컬
failed_vals = data_df[data_df['Status'] == 'Failed']['Bioactivities'].dropna() # 엎음

u_stat, p_val = stats.mannwhitneyu(clinical_vals, failed_vals, alternative='two-sided')

print(f"Mann-Whitney U statistic: {u_stat}")
print(f"P-value: {p_val}")

if p_val < 0.05:
    print("결과: 두 그룹 간에 통계적으로 유의미한 차이가 있습니다! (다른 놈임)")
else:
    print("결과: P-value가 0.05보다 큽니다. 두 그룹은 통계적으로 '그놈이 그놈'입니다.")

### AlogP

In [ ]:
data_df.groupby('Status')['AlogP'].mean() # 평균

In [ ]:
# boxplot
sns.boxplot(data_df, x = 'Status', y = 'AlogP', hue = 'Status')
sns.swarmplot(data=data_df, x='Status', y='AlogP', color='red', alpha=0.5)
plt.title('AlogP by Status')
plt.show()

- 승인된 약물은 전체적으로 뚜껑 아니면 바닥인데 임상 진행중인 약물들에 이상치가 보인다. 
- 이거 logP가 너무 높아도 안될건데... (너무 기름져서 물에 안녹음)

### 이상치 누구냐

In [ ]:
# 이상치를 찾아라 
idx_list = data_df.query('AlogP >= 5').index # 2, 13, 21, 57
for idx in idx_list:
    display(data_df[['Name','Molecular Weight','Molecular Formula', 'Smiles','AlogP', 'Max Phase']].loc[idx])

- VEDAPROFEN은 동물용 소염진통제다. 
- TIOXAPROFEN은 항진균제인데, 바르는 용도다. 음... 그럼 확실히 피부를 통해서 인지질 이중층을 공략하는쪽이 더 낫겠군. 

## Withdrawn Flag
- 물론 약에는 부작용이 따른다. 그래서 용법과 용량을 준수해야 하는 거고. 

In [ ]:
data_df.groupby(['Status','Withdrawn Flag']).size()

### 부작용 누구냐

In [ ]:
# 이상치를 찾아라 
idx_list = data_df.query('`Withdrawn Flag` == 1').index # 42, 56, 61, 62
for idx in idx_list:
    display(data_df[['Name','Molecular Weight','Molecular Formula', 'Smiles', 'Max Phase']].loc[idx])

#### 부작용
- Benoxaprofen: 간 괴사, 광과민성, 담즙정체성 황달
- Suprofen: 신장 독성 (옆구리 통증 증후군)
- Pirprofen: 심각한 간독성
- Indoprofen: 위장관 출혈 및 발암성 의혹

## HBA, HBD 분포

In [ ]:
data_df.groupby(['Status','HBA','HBD']).size() # 분포 왜이래요

In [ ]:
# FacetGrid를 사용해 Status별로 나눠서 보기
g = sns.FacetGrid(data_df, col="Status", height=4, aspect=1.2, col_order=['Approved', 'Clinical', 'Failed'])
g.map_dataframe(sns.histplot, x="HBA", y="HBD", discrete=(True, True), cbar=True, cmap="YlGnBu")

g.set_axis_labels("HBA (Acceptors)", "HBD (Donors)")
g.set_titles("{col_name} Group")
plt.tight_layout()
plt.show()

- 클리니컬은 뭘 많이 시도해서 그런가? 

## RO5 violation

In [ ]:
data_df.groupby(['Status','#RO5 Violations']).size() # 분포 왜이래요

### 위반하신분 누구시죠

In [ ]:
# 이상치를 찾아라 
# 쿼리썼더니 에러나데요... 
idx_list = data_df[data_df['#RO5 Violations'] == 1].index # 2, 13, 21, 57
idx_list
for idx in idx_list:
    display(data_df[['Name','Molecular Weight','Molecular Formula', 'Smiles','HBA','HBD','AlogP']].loc[idx])

- 뭐가 문제인가 했더니 RO5가 문제였네. 

# 결과

1. 이부프로펜(원조)이 활성에 있어서는 짱이다. 
2. RO5를 위반하지 않고 승인되더라도 부작용때문에 시장에서 퇴출되는 약들이 있다. 

## 이상치 친구들

1. 분자량: fenoprofen calcium
2. 생리활성: Ibuprofen (압도적)
3. AlogP: Lobuprofen, Frabuprofen, Tioxaprofen, Vedaprofen
4. Withdrawn flag: Benoxaprofen, Surprofen, Pirprofen, Indoprofen
    - 이상치까지는 아니고 플래그 선 애들
5. RO5: Lobuprofen, Frabuprofen, Tioxaprofen, Vedaprofen
    - AlogP때문에…